# Day 1 — 클라우드 대형 모델 smoke test (35B / 122B / 397B 공용)

**담당: 진영(35B) · 민우(122B) · 석웅(397B). RunPod 등 클라우드 GPU.**

세 명이 하는 일이 완전히 같고 모델만 다르므로 노트북을 3개로 나누지 않는다 —
**맨 아래 코드 셀에서 `MODEL_KEY` 한 줄만 자기 담당으로 바꿔서 쓴다.**
그래야 세 결과를 나중에 같은 스키마로 바로 비교할 수 있다.

## MODEL_KEY — 여기 한 줄만 바꾼다

In [ ]:
MODEL_KEY = "35b"  # "35b" | "122b" | "397b"

각 키가 어떤 `configs/experiments/*.yaml`에 매핑되는지:

| MODEL_KEY | config | 비고 |
|---|---|---|
| 35b | `35b_qdora_r8_1536` | resource: runpod_b300x1, NF4 약 23GB |
| 122b | `122b_qlora_r8_crop` | resource: runpod_b300x1, NF4 약 65GB |
| 397b | `397b_base_1536` | resource: runpod_b300x2, NF4 약 203GB |

**397B는 `397b_qdora_r8_1536`/`397b_qlora_r8_1536`(학습, B300x2 몇 시간 단위 비용)이 아니라
`397b_base_1536`(zero-shot ceiling test, 학습 없음)을 쓴다** — Day 1 smoke test 노트북에서
실수로 고비용 장시간 학습 job을 트리거하지 않기 위한 의도적 선택이다. 397B 학습 자체는
Day 1 스크리닝이 끝난 뒤 별도로 판단한다.

이 노트북은 각 config의 `task`(train/infer) 필드와 무관하게 **항상 `infer` 커맨드로
zero-shot smoke test만 수행한다** — 실제 PEFT 학습은 여기서 하지 않는다.

In [ ]:
MODEL_CONFIG = {
    "35b": "35b_qdora_r8_1536",
    "122b": "122b_qlora_r8_crop",
    "397b": "397b_base_1536",
}[MODEL_KEY]
print(f"MODEL_KEY={MODEL_KEY} -> config={MODEL_CONFIG}")

## 0. 스키마 대기 게이트

In [ ]:
import sys
import yaml
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

schema_report = REPO_ROOT / "data_meta" / "schema_report.md"
base_cfg = yaml.safe_load((REPO_ROOT / "configs" / "base.yaml").read_text(encoding="utf-8"))
columns = (base_cfg.get("data", {}) or {}).get("columns", {}) or {}

if not (schema_report.exists() and columns.get("image") is not None):
    print("\n" + "=" * 60)
    print("김진영의 스키마 확정을 기다리는 중입니다. MM 확인 후 다시 실행하세요.")
    print("=" * 60)
    raise SystemExit("스키마 대기 중")
print("스키마 확정 확인됨 — 계속 진행합니다.")

## 1. 인스턴스 환경 확인

선택한 모델의 NF4 추정치가 이 인스턴스에 들어가는지 먼저 계산해서 보여주고,
안 들어가면 여기서 멈춘다.

In [ ]:
import shutil, subprocess

def nvidia_smi_query(fields):
    out = subprocess.run(
        ["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True, timeout=10,
    ).stdout.strip()
    return [line.split(", ") for line in out.splitlines()]

gpu_rows = nvidia_smi_query("name,memory.total")
total_vram_gb = sum(float(r[1]) for r in gpu_rows) / 1024
print(f"GPU {len(gpu_rows)}장: {[r[0] for r in gpu_rows]}")
print(f"총 VRAM: {total_vram_gb:.1f} GB")

disk = shutil.disk_usage("/")
disk_free_gb = disk.free / 1e9
print(f"디스크 여유: {disk_free_gb:.1f} GB")

NF4_ESTIMATE_GB = {"35b": 23, "122b": 65, "397b": 203}[MODEL_KEY]
print(f"\n{MODEL_KEY} NF4 추정 VRAM: {NF4_ESTIMATE_GB} GB")

if NF4_ESTIMATE_GB > total_vram_gb:
    raise SystemExit(
        f"*** 이 인스턴스({total_vram_gb:.1f}GB)에 {MODEL_KEY}(추정 {NF4_ESTIMATE_GB}GB)가 안 들어갑니다. ***\n"
        "*** configs/resources/*.yaml에서 더 큰 인스턴스로 다시 확인하세요. ***"
    )
print(f"OK — {total_vram_gb:.1f}GB >= {NF4_ESTIMATE_GB}GB 추정치")

## 2. 가중치 확보 — 영속 캐시 경로에 받는다

파드를 다시 띄울 때마다 재다운로드하면 그 자체가 비용이다. **network volume 등 영속
경로**에 `HF_HOME`을 잡는다 — 컨테이너 기본 캐시(`~/.cache/huggingface`)는 파드가
내려가면 사라진다. 받기 전 디스크 여유를 확인하고, 시간 제한 없이 받는다
(측정 타임아웃 밖 — 397B는 800GB 안팎이라 몇 시간 걸릴 수 있다).

In [ ]:
import os

# RunPod network volume은 보통 /workspace 또는 /runpod-volume에 마운트된다.
# 이 인스턴스의 실제 마운트 경로로 바꿔서 쓸 것 — 기본 캐시 경로를 쓰면 파드를
# 내렸다 올릴 때마다 처음부터 다시 받아야 한다.
# .gitignore가 .hf_cache/ 를 제외 대상으로 이미 잡아뒀다 — 실수로 커밋 안 되게 이 이름을 그대로 쓴다
PERSISTENT_HF_HOME = os.environ.get("AI2_PERSISTENT_HF_HOME", "/workspace/.hf_cache")
os.environ["HF_HOME"] = PERSISTENT_HF_HOME
Path(PERSISTENT_HF_HOME).mkdir(parents=True, exist_ok=True)
print(f"HF_HOME = {PERSISTENT_HF_HOME}")

# 플레키한 대회장 망/클라우드 네트워크 대비 — 기본 10초는 자주 끊긴다
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"

MODEL_DISK_ESTIMATE_GB = {"35b": 70, "122b": 245, "397b": 800}[MODEL_KEY]
disk_free_gb = shutil.disk_usage(PERSISTENT_HF_HOME).free / 1e9
print(f"디스크 여유: {disk_free_gb:.1f} GB, 예상 필요: {MODEL_DISK_ESTIMATE_GB} GB")
if disk_free_gb < MODEL_DISK_ESTIMATE_GB:
    raise SystemExit(f"*** 디스크 여유 부족 — network volume 크기를 늘리세요. ***")

In [ ]:
from src.config import load_config
from huggingface_hub import snapshot_download

cfg = load_config(MODEL_CONFIG)
hf_id = cfg.get("model_config.hf_id") or cfg.model.get("hf_id")
print(f"모델: {hf_id}")
print("다운로드 시작 (시간 제한 없음 — 완료될 때까지 기다립니다)...")

snapshot_download(repo_id=hf_id, cache_dir=os.path.join(PERSISTENT_HF_HOME, "hub"))
print("다운로드 완료.")

## 3. Budget Guard 확인 ★ 돌리기 전에 반드시

특히 **연휴 당번**이 이걸 모르고 켜 두면 안 된다. 이 세션 하나가 예산의 몇 %를 쓸 수
있는지 미리 계산해서 보여준다.

In [ ]:
RESOURCE_NAME = cfg.get("resource")
resource_cfg = yaml.safe_load((REPO_ROOT / "configs" / "resources" / f"{RESOURCE_NAME}.yaml").read_text(encoding="utf-8"))

usd_per_hour = resource_cfg["cost"]["usd_per_hour"]
krw_per_hour = resource_cfg["cost"]["krw_per_hour"]
print(f"리소스: {RESOURCE_NAME}  (${usd_per_hour}/h = {krw_per_hour:,}원/h)")

rg = resource_cfg.get("budget_guard", {})
print(f"\n[리소스별 상한 — configs/resources/{RESOURCE_NAME}.yaml]")
print(f"  max_cost_krw: {rg.get('max_cost_krw'):,}" if rg.get('max_cost_krw') else "  max_cost_krw: (없음)")
print(f"  max_runtime_hours: {rg.get('max_runtime_hours')}")

env_max_cost = int(os.environ.get("AI2_MAX_COST_KRW", 0) or 0)
env_max_gpu = int(os.environ.get("AI2_MAX_GPU_COUNT", 0) or 0)
env_max_hours = float(os.environ.get("AI2_MAX_RUNTIME_HOURS", 0) or 0)
print(f"\n[.env 전역 상한]")
print(f"  AI2_MAX_COST_KRW: {env_max_cost:,}")
print(f"  AI2_MAX_GPU_COUNT: {env_max_gpu}")
print(f"  AI2_MAX_RUNTIME_HOURS: {env_max_hours}")

effective_max_cost = min(v for v in [rg.get("max_cost_krw"), env_max_cost] if v) if (rg.get("max_cost_krw") or env_max_cost) else None
if effective_max_cost:
    planned_hours = float(input("이 세션에서 계획한 예상 사용 시간(시간 단위)을 입력하세요: ") or "1")
    planned_cost = planned_hours * krw_per_hour
    pct = 100 * planned_cost / effective_max_cost
    print(f"\n예상 비용: {planned_cost:,.0f}원 ({planned_hours}h x {krw_per_hour:,}원/h)")
    print(f"적용 상한: {effective_max_cost:,}원")
    print(f"*** 이 세션은 예산의 {pct:.1f}%를 씁니다 ***")
    if pct > 100:
        raise SystemExit("*** 예산 상한 초과 — 계획한 시간을 줄이거나 매니저 승인을 받으세요. ***")
else:
    print("\n예산 상한이 설정돼 있지 않습니다 — .env의 AI2_MAX_COST_KRW를 채우는 걸 권장합니다.")

## 4. smoke test

**9B baseline과 반드시 같은 300문항 subset을 쓴다.** `subset()`은
`seed`(고정값 1, `configs/base.yaml`)와 샘플 id만으로 결정되는 해시 기반
선택이라, `--set subset.size=300 --set subset.stratify_by=type`을 그대로 쓰면
**모델이 달라도 자동으로 같은 300문항이 뽑힌다** — 별도 subset 파일을 안 만들어도 된다
(단, `seed`를 개별 config에서 덮어쓰지 않았는지만 확인).

**실패해도 예외를 던지고 끝내지 않는다.** exit≠0/OOM이면 `notes` 컬럼에 마지막 400자를
남기고 `data_meta/day1_cloud_probe_results.csv`에 append한다 — 파드가 비싼 GPU를 켠
채로 원인 불명 실패를 겪었을 때 그 흔적이 셀 출력에만 남고 사라지는 걸 막기 위해서다
(day1_baseline_9b.ipynb에서 같은 문제로 원인을 못 잡은 적이 있다).

In [ ]:
import csv, json, sys, time

def existing_run_dirs():
    runs_dir = REPO_ROOT / "runs"
    return set(runs_dir.iterdir()) if runs_dir.exists() else set()

def _classify_status(returncode, output):
    if returncode == 0:
        return "ok"
    if "OutOfMemoryError" in output or "CUDA out of memory" in output:
        return "OOM"
    return f"ERROR(exit={returncode})"

CLOUD_RESULTS_CSV = REPO_ROOT / "data_meta" / "day1_cloud_probe_results.csv"
CLOUD_RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
CLOUD_FIELDS = ["model_key", "config", "status", "notes", "accuracy", "peak_vram_gb",
                "sec_per_sample", "visual_tokens_median", "elapsed_hours", "cost_krw", "run_id"]

def _append_cloud_row(row):
    is_new = not CLOUD_RESULTS_CSV.exists()
    with open(CLOUD_RESULTS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CLOUD_FIELDS)
        if is_new:
            w.writeheader()
        w.writerow({k: row.get(k, "") for k in CLOUD_FIELDS})
        f.flush(); os.fsync(f.fileno())

before = existing_run_dirs()
args = ["infer", "-c", MODEL_CONFIG, "--split", "dev",
        "--set", "subset.size=300", "--set", "subset.stratify_by=type"]
t0 = time.time()
proc = subprocess.run([sys.executable, "-m", "src.run", *args],
                      cwd=str(REPO_ROOT), capture_output=True, text=True)
elapsed_hours = (time.time() - t0) / 3600
output = (proc.stdout or "") + (proc.stderr or "")
status = _classify_status(proc.returncode, output)
after = existing_run_dirs()
run_dir = sorted(after - before)[-1] if (after - before) else None
actual_cost_krw = elapsed_hours * krw_per_hour

if status != "ok":
    print(output[-3000:])
    _append_cloud_row({"model_key": MODEL_KEY, "config": MODEL_CONFIG, "status": status,
                       "notes": output[-400:], "elapsed_hours": round(elapsed_hours, 3),
                       "cost_krw": round(actual_cost_krw), "run_id": run_dir.name if run_dir else ""})
    print(f"\n*** {status} — data_meta/day1_cloud_probe_results.csv에 notes와 함께 기록했습니다. ***")
    result = None
else:
    summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
    result = summary["infer"]
    vt = result.get("visual_tokens") or {}
    _append_cloud_row({"model_key": MODEL_KEY, "config": MODEL_CONFIG, "status": status, "notes": "",
                       "accuracy": result.get("accuracy"), "peak_vram_gb": result.get("peak_vram_gb"),
                       "sec_per_sample": result.get("sec_per_sample"), "visual_tokens_median": vt.get("median"),
                       "elapsed_hours": round(elapsed_hours, 3), "cost_krw": round(actual_cost_krw),
                       "run_id": run_dir.name})
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print(f"\nrun_dir: {run_dir}")
    print(f"실제 소요: {elapsed_hours*60:.1f}분 ({elapsed_hours:.3f}h)")
    print(f"실제 비용: {actual_cost_krw:,.0f}원")

## 5. 결과 + NetGain 준비

predictions를 저장해두면(이미 `infer.save_logits=True` 기본값으로 `predictions.jsonl`에
logit까지 저장돼 있다) 나중에 9B baseline 대비 fixed/broken/net을 계산할 수 있다
(CLAUDE.md 147행: NetGain = 추가 GPU 비용당 추가로 맞힌 문제 수).

In [ ]:
if result is None:
    raise SystemExit(
        "4절이 실패해서 5절을 건너뜁니다 — "
        "data_meta/day1_cloud_probe_results.csv의 notes 컬럼에서 원인을 먼저 확인하세요."
    )

gpu_hours = elapsed_hours * len(gpu_rows)
vt = result.get("visual_tokens") or {}

print(f"| 항목 | 값 |")
print(f"|---|---|")
print(f"| model_key | {MODEL_KEY} |")
print(f"| config | {MODEL_CONFIG} |")
print(f"| accuracy | {result.get('accuracy')} |")
print(f"| visual_tokens(median) | {vt.get('median')} |")
print(f"| peak_vram_gb | {result.get('peak_vram_gb')} |")
print(f"| sec_per_sample | {result.get('sec_per_sample')} |")
print(f"| gpu_hours | {gpu_hours:.3f} |")
print(f"| cost_krw | {actual_cost_krw:,.0f} |")
print(f"| predictions | {run_dir}/predictions.jsonl |")

print("\n이 표와 predictions.jsonl 경로를 팀에 공유하면 9B baseline과 NetGain 비교가 가능합니다.")